In [ ]:
API TESTING:

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from google import genai

# 1. Provide the exact raw string path to your .env file
env_path = Path(r"C:\Users\msn\Desktop\Bon_Voyage_Pakistan_Qoder\testingFiles\.env")

print(f"Target path: {env_path}")
print(f"Exists on disk: {env_path.exists()}")

if not env_path.exists():
    raise FileNotFoundError(f"File not found at: {env_path}")

# 2. Load the variables from the specified .env file
load_dotenv(dotenv_path=env_path)

# 3. Retrieve the key (checking common variable names)
api_key = (
    os.getenv("GEMINI_API_KEY")
    or os.getenv("GoogleAPI")
    or os.getenv("GOOGLE_API_KEY")
)

if not api_key:
    raise ValueError(
        f"The .env file was found, but no API key was detected inside it. "
        f"Make sure your .env contains a line like: GEMINI_API_KEY=AIzaSy..."
    )

print(f"API key successfully loaded (Starts with: {api_key[:6]}...)")

# 4. Initialize client & verify connection
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-3.6-flash", contents="Respond with: Connection verified."
)

print("\n--- Test Response ---")
print(response.text.strip())

## Image Testing

In [ ]:
!pip install matplotlib

In [ ]:
import os
from google.genai import types
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
import matplotlib.pyplot as plt
from PIL import Image
from pydantic import BaseModel, Field

# 1. Load your .env file
env_path = Path(r"C:\Users\msn\Desktop\Bon_Voyage_Pakistan_Qoder\testingFiles\.env")
load_dotenv(dotenv_path=env_path)

api_key = (
    os.getenv("GEMINI_API_KEY")
    or os.getenv("GoogleAPI")
    or os.getenv("GOOGLE_API_KEY")
)

if not api_key:
    raise ValueError("API Key not found in .env file.")

client = genai.Client(api_key=api_key)


# 2. Define the exact response structure
class LandscapeLandmarkAnalysis(BaseModel):
    is_landmark_or_tourist_site: bool = Field(
        description="True if the image is a verified natural landscape, tourist attraction, or historical site in Pakistan. False if it's a random house, street, or modern generic building."
    )
    name: str = Field(
        default="Unknown",
        description="Official name of the place, valley, lake, fort, or landmark.",
    )
    location: str = Field(
        default="Unknown",
        description="City, District, Province/Region (e.g. Gilgit-Baltistan, KPK, Punjab, Sindh, Balochistan).",
    )
    landscape_type: str = Field(
        default="Unknown",
        description="Category (e.g. Mountain Lake, Historical Fort, Valley, Monument, Mosque, Pass).",
    )
    historical_background: str = Field(
        default="",
        description="Detailed historical origin, who built it or geological/natural history.",
    )
    why_famous: str = Field(
        default="",
        description="Key reasons why it is a prominent tourist spot or heritage location.",
    )


# 3. Path to your test image (replace with your file name/path)
image_path = "test_image_2.jfif"  # Place your picture in the same directory or provide full path

if not Path(image_path).exists():
    raise FileNotFoundError(
        f"Could not find image at {image_path}. Please place an image there."
    )

image = Image.open(image_path)

# Display the image inside the notebook
plt.figure(figsize=(6, 4))
plt.imshow(image)
plt.axis("off")
plt.show()

# 4. Prompt for recognition and strict filtering
prompt = """
Analyze this image carefully.
Determine whether it is a recognized natural landscape, tourist destination, or historic/cultural landmark in Pakistan.

Rules:
1. If this is a generic house, private residential building, normal street, or unidentified modern structure, set 'is_landmark_or_tourist_site' to False.
2. If it is a recognized tourist site or historic landmark, set 'is_landmark_or_tourist_site' to True, and provide the official name, exact location, landscape type, comprehensive background history, and why it is famous.
"""

# # 5. Execute Multimodal Request
# response = client.models.generate_content(
#     model="gemini-3.6-flash",
#     contents=[image, prompt],
#     config=types.GenerateContentConfig(
#         response_mime_type="application/json",
#         response_schema=LandscapeLandmarkAnalysis,
#     ),
# )
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=[image, prompt],
    config=types.GenerateContentConfig(
        tools=[{"google_search": {}}],  # Enables web retrieval
        response_mime_type="application/json",
        response_schema=LandscapeLandmarkAnalysis,
    ),
)

# 6. Parse and format the result
result = LandscapeLandmarkAnalysis.model_validate_json(response.text)

print("=" * 60)
print("              LANDSCAPE & LANDMARK SCAN RESULT              ")
print("=" * 60)
print(f"📍 Recognized Site:     {result.is_landmark_or_tourist_site}")
print(f"🏷️  Name:                {result.name}")
print(f"🗺️  Location:            {result.location}")
print(f"🏞️  Type:                {result.landscape_type}")
print("-" * 60)
print(f"📜 Background History:\n{result.historical_background}")
print("-" * 60)
print(f"⭐ Why It's Famous:\n{result.why_famous}")
print("=" * 60)

---


## GROQ API

In [ ]:
!pip install groq python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load variables from .env
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError(
        "GROQ_API_KEY not found. "
        "Make sure your .env file exists and contains the API key."
    )

print("API key loaded successfully.")
print(f"Testing model: qwen/qwen3.6-27b")

try:
    # Create Groq client
    client = Groq(api_key=api_key)

    # Send a test request
    completion = client.chat.completions.create(
        model="qwen/qwen3.6-27b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert Pakistan travel planner. "
                    "Give clear, practical, concise answers."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Plan a 5-day trip from Islamabad to Hunza. "
                    "The traveler likes mountains, photography, local food, "
                    "and cultural places. Give a day-by-day plan."
                ),
            },
        ],
        temperature=0.7,
        max_completion_tokens=1500,
    )

    result = completion.choices[0].message.content

    print("\n" + "=" * 60)
    print("API TEST SUCCESSFUL!")
    print("=" * 60)
    print("\nMODEL RESPONSE:\n")
    print(result)

except Exception as e:
    print("\n" + "=" * 60)
    print("API TEST FAILED!")
    print("=" * 60)
    print(f"\nERROR: {type(e).__name__}")
    print(f"DETAILS: {e}")

In [ ]:
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv
import edge_tts
from google import genai
from google.genai import types
from groq import Groq
from IPython.display import Audio, display
from pydantic import BaseModel, Field

# 1. Load keys safely from your backend .env file
env_path = Path(r"C:\Users\msn\Desktop\Bon_Voyage_Pakistan_Qoder\backend\.env")
load_dotenv(dotenv_path=env_path, override=True)
env_dict = dotenv_values(dotenv_path=env_path)

groq_key = (
    os.getenv("GROQ_API_KEY_TripPlan")
    or os.getenv("GROQ_API_KEY")
    or env_dict.get("GROQ_API_KEY_TripPlan")
)
gemini_key = (
    os.getenv("GEMINI_API_KEY")
    or os.getenv("GoogleAPI")
    or os.getenv("GOOGLE_API_KEY")
    or env_dict.get("GEMINI_API_KEY")
    or env_dict.get("GoogleAPI")
)

if not groq_key or not gemini_key:
    raise ValueError(
        f"Missing keys! Groq: {'OK' if groq_key else 'Missing'}, Gemini: {'OK' if gemini_key else 'Missing'}"
    )

# 2. Initialize Clients
groq_client = Groq(api_key=groq_key)
gemini_client = genai.Client(api_key=gemini_key)


# 3. Schema
class TranslationOutput(BaseModel):
    detected_language: str = Field(description="Detected source language.")
    english_translation: str = Field(description="Natural English translation.")
    urdu_translation: str = Field(
        description="Authentic Urdu translation in Nastaliq/Arabic script (اردو)."
    )


# 4. Processing Functions
def transcribe_audio_file(audio_path: str) -> str:
    print(f"\n[1/3] 🎙️ Transcribing audio file...")
    with open(audio_path, "rb") as file:
        transcription = groq_client.audio.transcriptions.create(
            file=(Path(audio_path).name, file.read()),
            model="whisper-large-v3",
            response_format="json",
        )
    return transcription.text


def translate_to_en_and_ur(text: str) -> TranslationOutput:
    print(f"\n[2/3] 🌐 Translating text via Gemini...")
    prompt = f"""
    You are a multilingual translator for travelers in Pakistan.
    Analyze the following input text:
    "{text}"

    1. Detect the source language.
    2. Provide an accurate and natural English translation.
    3. Provide an authentic Urdu translation in proper Urdu script (اردو).
    """
    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=TranslationOutput,
        ),
    )
    return TranslationOutput.model_validate_json(response.text)


async def generate_speech_file(
    text: str, voice: str, filename: str
) -> str:
    if not text.strip():
        return ""
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(filename)
    return filename


# 5. Master Pipeline
async def run_translator_pipeline(
    input_text: str = None, audio_path: str = None
):
    print("=" * 65)
    print("        MULTILINGUAL TRANSLATOR & SPEECH ENGINE       ")
    print("=" * 65)

    if audio_path and Path(audio_path).exists():
        raw_text = transcribe_audio_file(audio_path)
        print(f"✅ Original Transcript: \"{raw_text}\"")
    elif input_text:
        raw_text = input_text
        print(f"📝 Text Input: \"{raw_text}\"")
    else:
        raw_text = "Bonjour! Comment puis-je aller au fort de Lahore?"
        print(f"ℹ️ Default sample: \"{raw_text}\"")

    # Translate
    result = translate_to_en_and_ur(raw_text)

    print("\n" + "-" * 65)
    print(f"🌍 Detected Language:   {result.detected_language}")
    print(f"🇬🇧 English Translation: {result.english_translation}")
    print(f"🇵🇰 Urdu Translation:    {result.urdu_translation}")
    print("-" * 65)

    # Generate Neural Audio
    print("\n[3/3] 🔊 Synthesizing Audio (TTS)...")
    en_audio = "output_en.mp3"
    ur_audio = "output_ur.mp3"

    await generate_speech_file(
        result.english_translation, "en-US-ChristopherNeural", en_audio
    )
    await generate_speech_file(
        result.urdu_translation, "ur-PK-UzmaNeural", ur_audio
    )

    print(f"✅ English audio: {Path(en_audio).resolve()}")
    print(f"✅ Urdu audio:    {Path(ur_audio).resolve()}")
    print("=" * 65)

    # Inline Audio Players in Notebook
    print("\n🎧 Listen to English:")
    display(Audio(en_audio))
    print("🎧 Listen to Urdu:")
    display(Audio(ur_audio))


# 6. Direct Jupyter Execution
audio_target = r"C:\Users\msn\Desktop\Bon_Voyage_Pakistan_Qoder\testingFiles\classroom-german.mp3"
await run_translator_pipeline(audio_path=audio_target)

In [ ]:
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv
import edge_tts
from google import genai
from google.genai import types
from groq import Groq
from IPython.display import Audio, display
from pydantic import BaseModel, Field

# ==============================================================================
# 🛠️ CONFIGURATION & FILE PATHS (EDIT HERE FOR FUTURE TESTING)
# ==============================================================================

# 1. Path to your .env file
ENV_PATH = Path(r"C:\Users\msn\Desktop\Bon_Voyage_Pakistan_Qoder\backend\.env")

# 2. Path to the audio file you want to test
#    👉 CHANGE THIS PATH ANY TIME YOU HAVE A NEW AUDIO FILE:
TEST_AUDIO_FILE = Path(
    r"C:\Users\msn\Desktop\Bon_Voyage_Pakistan_Qoder\testingFiles\clinstructions.mp3"
)

# 3. Output directory for generated MP3 files
OUTPUT_DIR = Path(
    r"C:\Users\msn\Desktop\Bon_Voyage_Pakistan_Qoder\testingFiles"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 🔑 CLIENT INITIALIZATION
# ==============================================================================

load_dotenv(dotenv_path=ENV_PATH, override=True)
env_dict = dotenv_values(dotenv_path=ENV_PATH)

groq_key = (
    os.getenv("GROQ_API_KEY_TripPlan")
    or os.getenv("GROQ_API_KEY")
    or env_dict.get("GROQ_API_KEY_TripPlan")
)
gemini_key = (
    os.getenv("GEMINI_API_KEY")
    or os.getenv("GoogleAPI")
    or os.getenv("GOOGLE_API_KEY")
    or env_dict.get("GEMINI_API_KEY")
    or env_dict.get("GoogleAPI")
)

if not groq_key or not gemini_key:
    raise ValueError(
        f"Missing API keys! Groq: {'✅ Found' if groq_key else '❌ Missing'}, "
        f"Gemini: {'✅ Found' if gemini_key else '❌ Missing'}"
    )

groq_client = Groq(api_key=groq_key)
gemini_client = genai.Client(api_key=gemini_key)


# ==============================================================================
# 📋 STRUCTURED OUTPUT SCHEMA
# ==============================================================================


class TranslationResult(BaseModel):
    detected_source_language: str = Field(
        description="Source language detected (e.g., German, French, Arabic, Chinese, Spanish)."
    )
    english_translation: str = Field(
        description="Natural, accurate English translation."
    )
    urdu_translation: str = Field(
        description="Authentic Urdu translation in Nastaliq / Arabic script (اردو)."
    )


# ==============================================================================
# ⚙️ CORE PROCESSING FUNCTIONS
# ==============================================================================


def transcribe_voice(file_path: Path) -> str:
    """Extracts raw text from any audio file using Groq Whisper."""
    print(f"\n[1/3] 🎙️ Transcribing: {file_path.name}...")
    with open(file_path, "rb") as f:
        transcription = groq_client.audio.transcriptions.create(
            file=(file_path.name, f.read()),
            model="whisper-large-v3",
            response_format="json",
        )
    return transcription.text.strip()


def translate_text(input_text: str) -> TranslationResult:
    """Translates text to English and Urdu using Gemini 3.6 Flash."""
    print(f"\n[2/3] 🌐 Translating via Gemini 3.6 Flash...")
    prompt = f"""
    You are a multilingual translator for travelers and tourists in Pakistan.
    Analyze the following input text:
    "{input_text}"

    Tasks:
    1. Detect the source language.
    2. Provide an accurate and natural English translation.
    3. Provide an authentic, grammatically correct Urdu translation in proper Urdu script (اردو).
    """
    response = gemini_client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=TranslationResult,
        ),
    )
    return TranslationResult.model_validate_json(response.text)


async def synthesize_voice(text: str, voice: str, output_path: Path) -> Path:
    """Generates an MP3 audio file using Edge-TTS neural voices."""
    if not text.strip():
        return output_path
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(str(output_path))
    return output_path


# ==============================================================================
# 🚀 TEST RUNNER FUNCTIONS
# ==============================================================================


async def test_audio_translation(audio_path: Path):
    """Pipeline 1: Audio Input -> STT -> Translation -> Audio Files"""
    print("=" * 70)
    print("                 PIPELINE 1: AUDIO FILE TRANSLATION                 ")
    print("=" * 70)

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found at: {audio_path}")

    # 1. Transcribe Audio
    raw_transcript = transcribe_voice(audio_path)
    print(f"📝 Original Transcript:\n   \"{raw_transcript}\"")

    # 2. Translate to English & Urdu
    result = translate_text(raw_transcript)

    print("\n" + "-" * 70)
    print(f"🌍 Detected Language:   {result.detected_source_language}")
    print(f"🇬🇧 English Translation: {result.english_translation}")
    print(f"🇵🇰 Urdu Translation:    {result.urdu_translation}")
    print("-" * 70)

    # 3. Generate Spoken Output Files
    print("\n[3/3] 🔊 Synthesizing Neural Audio Files...")
    en_audio = OUTPUT_DIR / f"{audio_path.stem}_translated_english.mp3"
    ur_audio = OUTPUT_DIR / f"{audio_path.stem}_translated_urdu.mp3"

    await synthesize_voice(
        result.english_translation, "en-US-ChristopherNeural", en_audio
    )
    await synthesize_voice(
        result.urdu_translation, "ur-PK-UzmaNeural", ur_audio
    )

    print(f"✅ Saved English Voice: {en_audio}")
    print(f"✅ Saved Urdu Voice:    {ur_audio}")
    print("=" * 70)

    # Play inline in Jupyter
    try:
        print("\n🎧 Spoken English:")
        display(Audio(str(en_audio)))
        print("🎧 Spoken Urdu:")
        display(Audio(str(ur_audio)))
    except Exception:
        pass


async def test_text_translation(
    user_text: str, output_prefix: str = "text_test"
):
    """Pipeline 2: User Typed Text -> Translation -> Audio Files"""
    print("=" * 70)
    print("                 PIPELINE 2: TEXT INPUT TRANSLATION                 ")
    print("=" * 70)
    print(f"📥 Input Text: \"{user_text}\"")

    # 1. Translate
    result = translate_text(user_text)

    print("\n" + "-" * 70)
    print(f"🌍 Detected Language:   {result.detected_source_language}")
    print(f"🇬🇧 English Translation: {result.english_translation}")
    print(f"🇵🇰 Urdu Translation:    {result.urdu_translation}")
    print("-" * 70)

    # 2. Generate Audio Files
    print("\n🔊 Synthesizing Neural Audio Files...")
    en_audio = OUTPUT_DIR / f"{output_prefix}_english.mp3"
    ur_audio = OUTPUT_DIR / f"{output_prefix}_urdu.mp3"

    await synthesize_voice(
        result.english_translation, "en-US-ChristopherNeural", en_audio
    )
    await synthesize_voice(
        result.urdu_translation, "ur-PK-UzmaNeural", ur_audio
    )

    print(f"✅ Saved English Voice: {en_audio}")
    print(f"✅ Saved Urdu Voice:    {ur_audio}")
    print("=" * 70)

    # Play inline in Jupyter
    try:
        print("\n🎧 Spoken English:")
        display(Audio(str(en_audio)))
        print("🎧 Spoken Urdu:")
        display(Audio(str(ur_audio)))
    except Exception:
        pass



await test_audio_translation(TEST_AUDIO_FILE)

# Test with French, German, Arabic, Turkish, Chinese, etc.
sample_query = "Bonjour, où se trouve l'arrêt de bus pour aller à Murree?"
await test_text_translation(sample_query)

In [ ]:
import math
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv
import httpx
from pydantic import BaseModel, Field

# Load API Key
env_path = Path(r"C:\Users\msn\Desktop\Bon-Voyage-Pakistan\Bon_Voyage_Pakistan_Qoder\testingFiles\.env")
load_dotenv(dotenv_path=env_path, override=True)
env_dict = dotenv_values(dotenv_path=env_path)

GEOAPIFY_KEY = (
    os.getenv("GEOAPIFY_API_KEY")
    or env_dict.get("GEOAPIFY_API_KEY")
    or "YOUR_API_KEY"
)

# Coordinates for Islamabad Anchor
ISB_LAT = 33.6844
ISB_LNG = 73.0479


def calculate_haversine(lat1: float, lon1: float, lat2: float, lon2: float):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(math.radians(lat1))
        * math.cos(math.radians(lat2))
        * math.sin(dlon / 2) ** 2
    )
    dist_km = round(R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)), 1)
    eta_mins = max(3, int((dist_km / 25.0) * 60))
    return dist_km, eta_mins


def assign_hotel_badge_and_price(name: str, categories: list):
    name_lower = name.lower()
    cat_str = " ".join(categories).lower()

    if any(k in name_lower for k in ["pod", "capsule", "hostel", "dorm"]):
        return "Glamping / Pods", 9500, 4.5
    if (
        any(
            k in name_lower
            for k in ["serena", "marriott", "pearl continental", "luxury"]
        )
        or "5_star" in cat_str
    ):
        return "5-Star Luxury", 55000, 4.9
    if any(k in name_lower for k in ["haveli", "heritage", "palace"]):
        return "Heritage Stay", 28000, 4.8
    if any(k in name_lower for k in ["resort", "lodge", "view"]):
        return "Mountain Resort", 22000, 4.7
    return "Comfort Stay", 12000, 4.4


async def fetch_geoapify_hotels(
    lat: float, lon: float, radius_km: float = 15.0
):
    radius_meters = int(radius_km * 1000)
    # Geoapify circle format: filter=circle:lon,lat,radius_in_meters
    url = "https://api.geoapify.com/v2/places"
    params = {
        "categories": "accommodation.hotel,accommodation.motel,accommodation.guest_house,accommodation.hostel",
        "filter": f"circle:{lon},{lat},{radius_meters}",
        "bias": f"proximity:{lon},{lat}",
        "limit": 10,
        "apiKey": GEOAPIFY_KEY,
    }

    print(
        f"🔍 Querying Geoapify Places around ({lat}, {lon}) with radius {radius_km}km..."
    )
    async with httpx.AsyncClient(timeout=15.0) as client:
        response = await client.get(url, params=params)

    if response.status_code != 200:
        print(f"❌ Error {response.status_code}: {response.text}")
        return

    data = response.json()
    places = data.get("features", [])
    print(f"✅ Found {len(places)} stays from Geoapify!\n")

    print("=" * 70)
    print("                    HOTELS & STAYS FEED                    ")
    print("=" * 70)

    for place in places:
        props = place.get("properties", {})
        h_name = props.get("name") or props.get("formatted", "Hotel Stay")
        h_lat = props.get("lat")
        h_lon = props.get("lon")
        h_address = (
            props.get("address_line2")
            or props.get("street")
            or "Islamabad, Pakistan"
        )
        h_cats = props.get("categories", [])

        dist_km, eta_mins = calculate_haversine(lat, lon, h_lat, h_lon)
        badge, price_pkr, rating = assign_hotel_badge_and_price(h_name, h_cats)
        price_tag = f"PKR {int(price_pkr / 1000)}k"

        print(f"🏨 {h_name}")
        print(
            f"   🏷️ Badge: {badge} | 💰 PKR {price_pkr:,} (Pin: {price_tag}) | ★ {rating}"
        )
        print(
            f"   📍 Distance: {dist_km} km • ~{eta_mins} mins | Address: {h_address}"
        )
        print(f"   🗺️ Coordinates: ({h_lat}, {h_lon})")
        print("-" * 70)


# Run the test
await fetch_geoapify_hotels(ISB_LAT, ISB_LNG, radius_km=15.0)

In [ ]:
import os
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

env_path = Path(r"C:\Users\msn\Desktop\Bon-Voyage-Pakistan\Bon_Voyage_Pakistan_Qoder\testingFiles\.env")
load_dotenv(dotenv_path=env_path, override=True)
env_dict = dotenv_values(dotenv_path=env_path)

key_read = (
    os.getenv("GEOAPIFY_API_KEY")
    or env_dict.get("GEOAPIFY_API_KEY")
    or "NOT_FOUND"
)
print(f"Key loaded from file: {key_read}")
print(f"Key length: {len(key_read)} characters")

In [ ]:
from pathlib import Path
from dotenv import dotenv_values, load_dotenv

# 1. Define the exact path
env_path = Path(
    r"C:\Users\msn\Desktop\Bon-Voyage-Pakistan\Bon_Voyage_Pakistan_Qoder\testingFiles\.env"
)

print(f"Target path: {env_path}")
print(f"Directory exists: {env_path.parent.exists()}")
print(f"File exists: {env_path.exists()}\n")

# 2. Paste your 32-character Geoapify key here
YOUR_GEOAPIFY_KEY = "454af56eb7eb4805ab7fc12bd150891a"

# 3. Force write the key directly to this file
env_path.parent.mkdir(parents=True, exist_ok=True)
current_content = (
    env_path.read_text(encoding="utf-8") if env_path.exists() else ""
)
filtered_lines = [
    line
    for line in current_content.splitlines()
    if not line.startswith("GEOAPIFY_API_KEY")
]
filtered_lines.append(f"GEOAPIFY_API_KEY={YOUR_GEOAPIFY_KEY.strip()}")
env_path.write_text("\n".join(filtered_lines) + "\n", encoding="utf-8")

# 4. Reload and verify
load_dotenv(dotenv_path=env_path, override=True)
parsed_values = dotenv_values(dotenv_path=env_path)

loaded_key = parsed_values.get("GEOAPIFY_API_KEY")
print(f"✅ Successfully wrote and loaded key: {loaded_key}")

In [ ]:
import httpx

api_key = parsed_values.get("GEOAPIFY_API_KEY")

url = "https://api.geoapify.com/v2/places"
params = {
    "categories": "accommodation.hotel,accommodation.motel,accommodation.guest_house",
    "filter": "circle:73.0479,33.6844,35000",
    "bias": "proximity:73.0479,33.6844",
    "limit": 5,
    "apiKey": api_key,
}

response = httpx.get(url, params=params, timeout=15.0)
print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    features = response.json().get("features", [])
    print(f"✅ Found {len(features)} stays near Islamabad:")
    for place in features:
        props = place.get("properties", {})
        name = props.get("name") or props.get("formatted", "Hotel Stay")
        lat = props.get("lat")
        lon = props.get("lon")
        print(f"• {name} | Coordinates: ({lat}, {lon})")
else:
    print(f"❌ Error: {response.text}")

In [ ]:
import os
from pathlib import Path
import urllib.parse
from dotenv import dotenv_values, load_dotenv
import requests

# ---------------------------------------------------------
# 1. Load API Key from Environment
# ---------------------------------------------------------
env_path = Path(
    r"C:\Users\msn\Desktop\Bon-Voyage-Pakistan\Bon_Voyage_Pakistan_Qoder\testingFiles\.env"
)

if not env_path.exists():
    raise FileNotFoundError(f"Cannot find .env file at {env_path}")

load_dotenv(dotenv_path=env_path, override=True)
env_dict = dotenv_values(dotenv_path=env_path)

api_key = (
    os.getenv("Google_Places_API_Key")
    or env_dict.get("Google_Places_API_Key")
    or os.getenv("GOOGLE_PLACES_API_KEY")
    or env_dict.get("GOOGLE_PLACES_API_KEY")
)

if not api_key:
    raise ValueError(f"Google_Places_API_Key not found in {env_path}")

# ---------------------------------------------------------
# 2. Configure Coordinates and Search Radius
# ---------------------------------------------------------
# Default anchor for Islamabad: (33.6844, 73.0479)
user_lat = 33.6844
user_lng = 73.0479
radius_km = 15.0
radius_meters = radius_km * 1000.0

# ---------------------------------------------------------
# 3. Google Places API (New) - Nearby Search
# ---------------------------------------------------------
url = "https://places.googleapis.com/v1/places:searchNearby"

headers = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": api_key,
    "X-Goog-FieldMask": (
        "places.id,places.displayName,places.formattedAddress,"
        "places.rating,places.userRatingCount,places.location,places.types"
    ),
}

payload = {
    "includedTypes": ["hotel", "lodging", "resort_hotel", "guest_house", "motel"],
    "maxResultCount": 10,
    "locationRestriction": {
        "circle": {
            "center": {"latitude": user_lat, "longitude": user_lng},
            "radius": radius_meters,
        }
    },
}

print(f"🔍 Searching hotels within {radius_km} km of ({user_lat}, {user_lng}) using Places API (New)...\n")
response = requests.post(url, headers=headers, json=payload, timeout=15)

if response.status_code != 200:
    print(f"❌ Error {response.status_code}: {response.text}")
else:
    data = response.json()
    places = data.get("places", [])

    if not places:
        print("⚠️ No stays found within this radius.")
    else:
        print("=" * 80)
        print(f"               FOUND {len(places)} NEARBY HOTELS & STAYS")
        print("=" * 80)

        for idx, place in enumerate(places, 1):
            name = place.get("displayName", {}).get("text", "Unknown Hotel")
            rating = place.get("rating", "N/A")
            reviews = place.get("userRatingCount", 0)
            address = place.get("formattedAddress", "Address not available")
            loc = place.get("location", {})
            h_lat = loc.get("latitude")
            h_lng = loc.get("longitude")

            if h_lat and h_lng:
                maps_url = (
                    f"https://www.google.com/maps/dir/?api=1"
                    f"&origin={user_lat},{user_lng}"
                    f"&destination={h_lat},{h_lng}&travelmode=driving"
                )
            else:
                encoded_dest = urllib.parse.quote(f"{name}, {address}")
                maps_url = f"https://www.google.com/maps/dir/?api=1&destination={encoded_dest}"

            print(f"{idx}. 🏨 {name}")
            print(f"   ⭐ Rating: {rating} ({reviews} reviews)")
            print(f"   📍 Address: {address}")
            print(f"   🌐 Coordinates: ({h_lat}, {h_lng})")
            print(f"   🗺️ Directions: {maps_url}")
            print("-" * 80)

In [ ]:
import json
import os
from pathlib import Path
import random
import time
from dotenv import dotenv_values, load_dotenv
from google import genai
from PIL import Image

# 1. Load API Key
env_path = Path(
    r"C:\Users\msn\Desktop\Bon-Voyage-Pakistan\Bon_Voyage_Pakistan_Qoder\testingFiles\.env"
)
load_dotenv(dotenv_path=env_path, override=True)
env_dict = dotenv_values(dotenv_path=env_path)

GEMINI_KEY = (
    os.getenv("GEMINI_API_KEY")
    or env_dict.get("GEMINI_API_KEY")
    or os.getenv("GOOGLE_API_KEY")
    or env_dict.get("Google_Places_API_Key")
)

if not GEMINI_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env")


def analyze_landmark_image(image_path: str, api_key: str) -> dict:
    client = genai.Client(api_key=api_key)

    # Convert .jfif to standardized RGB
    img = Image.open(image_path).convert("RGB")

    prompt = """
    You are an expert cultural, historical, and architectural guide for Pakistan.
    Analyze this photo:
    1. Identify the Pakistani landmark, monument, fort, mosque, shrine, or scenic site shown.
    2. Provide its history, architectural significance, and interesting facts.

    Return ONLY a valid JSON object matching this schema:
    {
      "landmark_name": "Official name of the landmark",
      "city_or_region": "City, District, or Province in Pakistan",
      "historical_era": "Era or year constructed",
      "history_overview": "2-3 concise sentences detailing its origin, builder, and historical significance.",
      "interesting_facts": [
        "Fact 1",
        "Fact 2",
        "Fact 3"
      ],
      "travel_tip": "One practical tip for tourists visiting this spot."
    }
    Return raw JSON only. Do not wrap in markdown fences or commentary.
    """

    # Uncongested endpoints from your active key list
    priority_models = [
        "models/gemini-3.1-flash-image",
        "models/gemini-3-pro-image",
        "models/gemini-3.1-pro-preview",
        "models/gemini-3.5-flash-lite",
        "models/gemini-flash-latest-high-res-exp",
    ]

    for model_name in priority_models:
        for attempt in range(1, 3):
            try:
                print(f"🔄 Trying: {model_name} (Attempt {attempt})...")
                response = client.models.generate_content(
                    model=model_name,
                    contents=[img, prompt],
                )

                if response and response.text:
                    raw_text = response.text.strip()
                    if raw_text.startswith("```"):
                        raw_text = raw_text.split("\n", 1)[-1]
                    if raw_text.endswith("```"):
                        raw_text = raw_text.rsplit("\n", 1)[0]
                    raw_text = (
                        raw_text.replace("```json", "")
                        .replace("```", "")
                        .strip()
                    )

                    print(f"✅ Success with {model_name}!\n")
                    return json.loads(raw_text)

            except Exception as e:
                err = str(e)
                if "503" in err:
                    sleep_time = random.uniform(2.0, 4.0)
                    print(
                        f"⚠️ High load on {model_name}. Pausing {sleep_time:.1f}s..."
                    )
                    time.sleep(sleep_time)
                else:
                    print(f"⚠️ Skipped {model_name}: {err}")
                    break

    raise RuntimeError("All specialized vision models currently at capacity.")


# 3. Execution
if __name__ == "__main__":
    sample_image = r"C:\Users\msn\Desktop\Bon-Voyage-Pakistan\Bon_Voyage_Pakistan_Qoder\testingFiles\test_image_1.jfif"

    if not os.path.exists(sample_image):
        print(f"❌ File not found at: {sample_image}")
    else:
        print(f"📸 Uploading & analyzing: {Path(sample_image).name}...")
        details = analyze_landmark_image(sample_image, GEMINI_KEY)

        print("=" * 70)
        print(
            f"🏛️  {details.get('landmark_name')} ({details.get('city_or_region')})"
        )
        print(f"⏳ Built / Era: {details.get('historical_era')}")
        print("=" * 70)
        print(f"\n📖 Overview:\n{details.get('history_overview')}\n")
        print("💡 Interesting Facts:")
        for idx, fact in enumerate(details.get("interesting_facts", []), 1):
            print(f"  {idx}. {fact}")
        print(f"\n🎒 Traveler's Tip:\n{details.get('travel_tip')}")
        print("=" * 70)

In [1]:
import json
from datetime import datetime, timezone
import httpx

# Coordinates covering major tourism corridors
REGIONS = {
    "Islamabad_Rawalpindi": {"lat": 33.6844, "lon": 73.0479},
    "Hunza_Gilgit": {"lat": 36.3167, "lon": 74.6500},
    "Swat_Kalam": {"lat": 35.4907, "lon": 72.5833},
    "Skardu": {"lat": 35.2971, "lon": 75.6333},
}


# 1. Fetch Live Weather Alerts (Open-Meteo)
def fetch_weather_advisories():
    alerts = []
    print("🌧️ Fetching meteorological alerts from Open-Meteo...")

    for name, coords in REGIONS.items():
        url = (
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={coords['lat']}&longitude={coords['lon']}&"
            f"daily=weather_code,precipitation_probability_max,wind_speed_10m_max&timezone=auto"
        )
        try:
            res = httpx.get(url, timeout=5.0)
            if res.status_code == 200:
                daily = res.json().get("daily", {})
                precip = daily.get("precipitation_probability_max", [0])[0] or 0
                wind = daily.get("wind_speed_10m_max", [0])[0] or 0

                # High precipitation or wind triggers an alert card
                if precip >= 60 or wind >= 40:
                    severity = "danger" if (precip > 80 or wind > 60) else "warning"
                    alerts.append({
                        "id": f"wx-{name.lower()}",
                        "category": "weather",
                        "severity": severity,
                        "title": f"Adverse Weather Warning: {name.replace('_', ' ')}",
                        "region": name.replace("_", " "),
                        "summary": f"Rain probability {precip}% with wind gusts up to {wind} km/h.",
                        "detailed_advisory": "Drivers on winding and mountainous roads must stay alert for reduced visibility, slick asphalt, and localized ponding.",
                        "source": "Global Met / PMD Tracking",
                        "issued_at": datetime.now(timezone.utc).isoformat(),
                    })
        except Exception as e:
            print(f"⚠️ Weather check failed for {name}: {e}")

    return alerts


# 2. Fetch Disaster & Seismic Alerts (USGS & ReliefWeb)
def fetch_disaster_alerts():
    disasters = []
    print("🚨 Checking seismic and flood data (USGS / UN OCHA)...")

    # USGS Quake API for Pakistan bounding box (lat: 23 to 37, lon: 60 to 78)
    usgs_url = (
        "https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson"
        "&minmagnitude=4.5&minlatitude=23.5&maxlatitude=37.0"
        "&minlongitude=60.5&maxlongitude=77.5&limit=3"
    )
    try:
        res = httpx.get(usgs_url, timeout=6.0)
        if res.status_code == 200:
            features = res.json().get("features", [])
            for feat in features:
                props = feat.get("properties", {})
                place = props.get("place", "Pakistan Region")
                mag = props.get("mag", 0.0)
                disasters.append({
                    "id": f"eq-{feat.get('id')}",
                    "category": "disaster",
                    "severity": "danger" if mag >= 5.5 else "warning",
                    "title": f"M {mag:.1f} Earthquake Recorded",
                    "region": place,
                    "summary": f"Magnitude {mag} seismic event detected near {place}.",
                    "detailed_advisory": "Aftershocks are possible in hilly terrain. Verify bridge integrity and check for rockfall along valley routes before proceeding.",
                    "source": "USGS Early Warning",
                    "issued_at": datetime.now(timezone.utc).isoformat(),
                })
    except Exception as e:
        print(f"⚠️ USGS query error: {e}")

    return disasters


# 3. Roads & Mountain Passes (Curated National Corridor Feed)
def get_roads_and_passes():
    print("🛣️ Loading mountain passes and highway status feed...")
    return [
        {
            "id": "pass-babusar-n15",
            "category": "roads_passes",
            "severity": "warning",
            "title": "Babusar Pass (N-15) - Daylight Hours Only",
            "region": "Kaghan Valley - Chilas",
            "summary": "Pass is operational daily from 06:00 AM to 05:00 PM.",
            "detailed_advisory": "Nighttime passage is strictly barred past Naran checkpoint due to sub-zero drops, black ice, and thin emergency response coverage.",
            "source": "District Administration / NHMP",
            "issued_at": datetime.now(timezone.utc).isoformat(),
        },
        {
            "id": "kkh-dasu-n35",
            "category": "roads_passes",
            "severity": "info",
            "title": "Karakoram Highway (N-35) - Single Lane Open",
            "region": "Upper Kohistan (Dasu)",
            "summary": "Routine maintenance and rock clearing underway. Expect minor delays.",
            "detailed_advisory": "Follow on-site National Highway Authority marshals. Heavy transport vehicles are regulated during peak hours.",
            "source": "NHA Patrol",
            "issued_at": datetime.now(timezone.utc).isoformat(),
        },
        {
            "id": "safety-murree-express",
            "category": "public_safety",
            "severity": "info",
            "title": "Murree Expressway (E-75) Entry Advisory",
            "region": "Murree / Galyat",
            "summary": "Vehicle entry quota monitored at 17-Mile Toll Plaza on weekends.",
            "detailed_advisory": "Carry verified hotel reservations and valid mechanical fitness certificates. Strict parking controls active on Mall Road.",
            "source": "Traffic Police Murree",
            "issued_at": datetime.now(timezone.utc).isoformat(),
        },
    ]


if __name__ == "__main__":
    wx_data = fetch_weather_advisories()
    disaster_data = fetch_disaster_alerts()
    road_data = get_roads_and_passes()

    all_alerts = wx_data + disaster_data + road_data

    print("\n" + "=" * 75)
    print(f"          FETCHED {len(all_alerts)} LIVE NOTIFICATIONS & ADVISORIES")
    print("=" * 75)

    for alert in all_alerts:
        tag = alert["severity"].upper()
        print(f"\n[{tag}] ({alert['category'].upper()}) - {alert['region']}")
        print(f"📌 {alert['title']}")
        print(f"📝 {alert['summary']}")
        print(f"ℹ️  Advisory: {alert['detailed_advisory']}")
        print(f"🏛️  Source: {alert['source']}")
        print("-" * 75)

🌧️ Fetching meteorological alerts from Open-Meteo...
🚨 Checking seismic and flood data (USGS / UN OCHA)...
🛣️ Loading mountain passes and highway status feed...

          FETCHED 10 LIVE NOTIFICATIONS & ADVISORIES

[DANGER] (WEATHER) - Islamabad Rawalpindi
📌 Adverse Weather Warning: Islamabad Rawalpindi
📝 Rain probability 86% with wind gusts up to 13.9 km/h.
ℹ️  Advisory: Drivers on winding and mountainous roads must stay alert for reduced visibility, slick asphalt, and localized ponding.
🏛️  Source: Global Met / PMD Tracking
---------------------------------------------------------------------------

[WARNING] (WEATHER) - Hunza Gilgit
📌 Adverse Weather Warning: Hunza Gilgit
📝 Rain probability 78% with wind gusts up to 4.3 km/h.
ℹ️  Advisory: Drivers on winding and mountainous roads must stay alert for reduced visibility, slick asphalt, and localized ponding.
🏛️  Source: Global Met / PMD Tracking
---------------------------------------------------------------------------

[WARNING] (W

In [ ]:
import requests

def test_openweather_api(city_name, api_key):
    # Base URL for Current Weather Data API
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    
    # Query parameters: 'q' for city, 'appid' for key, and 'units' for Celsius
    params = {
        'q': city_name,
        'appid': api_key,
        'units': 'metric'
    }
    
    try:
        # Send a GET request to the OpenWeather API
        response = requests.get(base_url, params=params)
        
        # Check if the request was successful (Status Code 200)
        if response.status_code == 200:
            data = response.json()
            
            # Extract basic weather details
            city = data.get('name')
            country = data.get('sys', {}).get('country')
            temp = data.get('main', {}).get('temp')
            humidity = data.get('main', {}).get('humidity')
            description = data.get('weather', [{}])[0].get('description')
            
            print("✅ API Connection Successful!")
            print(f"Weather in {city}, {country}:")
            print(f" - Temperature: {temp}°C")
            print(f" - Humidity: {humidity}%")
            print(f" - Condition: {description.capitalize()}")
            
        elif response.status_code == 401:
            print("❌ Error 401: Invalid API Key. Please check your credentials.")
        elif response.status_code == 404:
            print(f"❌ Error 404: City '{city_name}' not found.")
        else:
            print(f"❌ Error {response.status_code}: {response.text}")
            
    except requests.exceptions.RequestException as e:
        print(f"🔗 Network Error: Could not connect to the API. Details: {e}")

if __name__ == "__main__":
    # ⚠️ Replace with your actual OpenWeather API key
    YOUR_API_KEY = ""
    CITY = "Islamabad"
    
    test_openweather_api(CITY, YOUR_API_KEY)


✅ API Connection Successful!
Weather in Islamabad, PK:
 - Temperature: 31.58°C
 - Humidity: 62%
 - Condition: Clear sky
